In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Jacob2020_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["UP-8036", "UP-8165", "UP-8167"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 20968 × 18839
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 968/968 [00:01<00:00, 753.04it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,180,0.858451,True,0.011806,0.018169,1.651291
AC092667.2,False,14,0.066768,True,0.000686,0.000907,1.292671
ZNF367,False,422,2.012591,True,0.025390,0.036237,1.406310
TRIM63,False,29,0.138306,True,0.002128,0.003759,1.893831
PABPN1P1,False,66,0.314765,True,0.003832,0.005219,1.326525
...,...,...,...,...,...,...,...
REELD1,False,21,0.100153,True,0.001379,0.002219,1.779598
LINC01775,False,11,0.052461,True,0.000568,0.000722,1.270593
LINC02338,False,16,0.076307,True,0.001032,0.001667,1.731135


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 20968 × 15417
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
metadata_data = {
    'Author': ['Jacob2020'] * 3,
    'donor_id': ["UP-8036", "UP-8165", "UP-8167"],
    'stage': ['Primary'] * 3,
    'assay': ['Drop-seq'] * 3,
    'tissue': [
        'right temporal lobe', 'left frontal lobe', 'left frontal lobe'
    ],
    'Cells': ['Total'] * 3,
    'Method': ['cell'] * 3
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

      Author donor_id    stage     assay               tissue  Cells Method
0  Jacob2020  UP-8036  Primary  Drop-seq  right temporal lobe  Total   cell
1  Jacob2020  UP-8165  Primary  Drop-seq    left frontal lobe  Total   cell
2  Jacob2020  UP-8167  Primary  Drop-seq    left frontal lobe  Total   cell


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id     Author    stage     assay               tissue  Cells  \
0      UP-8036  Jacob2020  Primary  Drop-seq  right temporal lobe  Total   
1      UP-8036  Jacob2020  Primary  Drop-seq  right temporal lobe  Total   
2      UP-8036  Jacob2020  Primary  Drop-seq  right temporal lobe  Total   
3      UP-8036  Jacob2020  Primary  Drop-seq  right temporal lobe  Total   
4      UP-8036  Jacob2020  Primary  Drop-seq  right temporal lobe  Total   
...        ...        ...      ...       ...                  ...    ...   
20963  UP-8167  Jacob2020  Primary  Drop-seq    left frontal lobe  Total   
20964  UP-8167  Jacob2020  Primary  Drop-seq    left frontal lobe  Total   
20965  UP-8167  Jacob2020  Primary  Drop-seq    left frontal lobe  Total   
20966  UP-8167  Jacob2020  Primary  Drop-seq    left frontal lobe  Total   
20967  UP-8167  Jacob2020  Primary  Drop-seq    left frontal lobe  Total   

      Method  
0       cell  
1       cell  
2       cell  
3       cell  
4       cell

In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
8036.Tumor.1-2-1,UP-8036,932,1494.113770,Neoplastic,Stem-like,NPC-like,Young neuron,Pluripotent Stem Cells,malignant cell
8036.Tumor.2-2-1,UP-8036,411,999.152527,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Oligodendrocytes,malignant cell
8036.Tumor.3-2-1,UP-8036,4049,2596.392578,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Oligodendrocytes,malignant cell
8036.Tumor.4-2-1,UP-8036,986,1576.269043,Neoplastic,Stem-like,OPC-like,Neuron,Neurons,malignant cell
8036.Tumor.5-2-1,UP-8036,1612,1824.406860,Neoplastic,Stem-like,NPC-like,Neuron,Neurons,malignant cell
...,...,...,...,...,...,...,...,...,...
8167.Tumor.12331-2-1,UP-8167,1293,1760.315796,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Neural Stem/Precursor Cells,malignant cell
8167.Tumor.12332-2-1,UP-8167,590,1195.298096,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Neural Stem/Precursor Cells,malignant cell
8167.Tumor.12333-2-1,UP-8167,1419,1788.369995,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Neural Stem/Precursor Cells,malignant cell
8167.Tumor.12334-2-1,UP-8167,841,1421.559448,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Neural Stem/Precursor Cells,malignant cell


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    770 total control genes are used. (0:00:00)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    601 total control genes are used. (0:00:00)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Jacob2020_Part3.h5ad")